# A Arquitetura de Alta Performance

1. O Motor de Dados (SQLite): É a escolha definitiva. Ele lê e escreve no disco em milissegundos. Se você tem um documento com 500 células de texto e edita apenas uma vírgula, o Rust vai alterar apenas 1 linha no banco de dados, sem reescrever o documento inteiro.

2. O Gerenciamento de Imagens (Caminho Local): Zero Base64 no banco de dados. O Rust vai pegar o arquivo bruto .png ou .jpg, salvar em uma pasta oculta do sistema (AppData\Local\SeuApp\Images) e o SQLite vai guardar apenas o endereço de texto (C:\...\imagem.png). Isso mantém o banco de dados na casa dos Kilobytes.

3. O Frontend (React/TS): O maior vilão da lentidão em apps web/desktop é a tela redesenhando coisas sem necessidade (re-renders). Vamos estruturar os componentes do React de forma que, se você estiver digitando na "Célula 3", as células 1 e 2 fiquem congeladas na memória, sem gastar processamento.

# O Passo a Passo Inicial (Foco no Windows)

Como você está no Windows, não precisamos instalar aqueles pacotes complexos do Linux. Você só precisa garantir que o ambiente nativo de compilação da Microsoft esteja pronto para o Rust fazer a mágica dele.

## Passo 1: Preparando a Máquina

Se ainda não tiver, instale apenas estes três:

- Node.js (versão LTS).

- Visual Studio C++ Build Tools (Baixe o instalador no site da Microsoft e marque a opção "Desenvolvimento para Desktop com C++". É obrigatório para compilar código Rust no Windows).

- Rust (Baixe e rode o rustup-init.exe do site oficial).

## Passo 2: O Primeiro Prompt Otimizado para sua IA

Copie e cole isso para o seu agente gerador de código para criar a fundação do projeto já com o SQLite na jogada:

"Atue como um Engenheiro de Software Sênior especialista em performance com Rust e Tauri. Estou no Windows (com Node, Rust e VS Build Tools já instalados). Quero inicializar um projeto Tauri novo usando React e TypeScript. Meu foco absoluto é ter um app desktop extremamente rápido e com uso mínimo de RAM.

- Me forneça o comando exato de terminal para inicializar esse projeto Tauri com o template React/TS.

- Nossa regra de arquitetura é usar o SQLite para ter altíssima performance de I/O, evitando salvar JSONs gigantes. Quais "crates" devo adicionar no Cargo.toml do Rust (ex: rusqlite e serde) e quais pacotes devo adicionar no frontend? Me dê apenas os comandos de instalação e as adições necessárias nos arquivos de configuração, sem explicações longas."

documents:

CREATE TABLE IF NOT EXISTS documents (
    id TEXT PRIMARY KEY,          -- UUID gerado no frontend
    title TEXT NOT NULL,          -- O título do documento
    created_at DATETIME DEFAULT CURRENT_TIMESTAMP,
    updated_at DATETIME DEFAULT CURRENT_TIMESTAMP
);

cells:
CREATE TABLE IF NOT EXISTS cells (
    id TEXT PRIMARY KEY,          -- UUID da célula
    document_id TEXT NOT NULL,    -- A qual documento essa célula pertence
    position REAL NOT NULL,       -- O pulo do gato para performance (Número Decimal)
    cell_type TEXT NOT NULL,      -- 'markdown', 'code' ou 'image'
    content TEXT NOT NULL,        -- O texto em si, o código, ou o CAMINHO da imagem (C:\...)
    metadata TEXT,                -- Um JSON super leve para guardar configurações extras
    created_at DATETIME DEFAULT CURRENT_TIMESTAMP,
    updated_at DATETIME DEFAULT CURRENT_TIMESTAMP,
    FOREIGN KEY (document_id) REFERENCES documents(id) ON DELETE CASCADE
);

-- Criar um índice para o Rust buscar as células na ordem certa em 0.001 segundos
CREATE INDEX IF NOT EXISTS idx_cells_doc_pos ON cells(document_id, position);

## Por que essa estrutura é de "Alta Performance"?


1. O Truque da Coluna position (REAL/Decimal)
Imagine que você tem a Célula 1 e a Célula 2. Você quer inserir um novo texto entre as duas.

- Jeito amador (Lento): Se as posições fossem 1, 2, 3... ao inserir uma no meio, o banco teria que reescrever a posição de todas as células abaixo dela (Update em 500 linhas).

- Nosso jeito (Rápido): A Célula 1 tem posição 1000.0. A Célula 2 tem 2000.0. Se você cria uma no meio, o frontend manda a posição 1500.0. O banco faz apenas 1 insert e a ordem continua perfeita. Custo de processamento quase zero.

2. A Coluna metadata (O Segredo das Imagens)
Como combinamos, o content da célula de imagem vai guardar apenas algo como C:\Users\Eric\AppData\Local\SeuApp\imagens\foto1.png.
Mas e as setinhas vermelhas e anotações do Fabric.js? Elas vão na coluna metadata. Você salva um JSON pequenininho contendo apenas as coordenadas do desenho ({"setas": [{"x": 10, "y": 20}]}). Quando a tela carregar, o frontend puxa a foto do HD e desenha as setinhas por cima usando esses dados.

Prompt para a sua IA (Fase do Banco de Dados)

Copie e cole este prompt para o seu gerador de código construir a ponte segura entre o Rust e o SQLite:

"Atue como um Especialista em Rust e SQLite. No meu projeto Tauri, preciso configurar o banco de dados local para alta performance usando a crate rusqlite.

1. Crie um arquivo db.rs no backend.

2. Escreva uma função de inicialização que crie um arquivo database.sqlite na pasta AppData local do usuário (use a crate tauri::api::path::app_data_dir).

3. Execute o código SQL de criação de duas tabelas: documents (id, title, created_at, updated_at) e cells (id, document_id, position REAL, cell_type, content, metadata, created_at, updated_at). Inclua o índice idx_cells_doc_pos.

4. Forneça o código Rust de forma modular e explique brevemente como chamar essa inicialização dentro da função main() do Tauri antes da janela do frontend abrir."

Quer que eu já prepare a estrutura visual dos Componentes React para exibir essas células na tela, ou prefere focar em gerar os comandos de CRUD (Insert/Update) no Rust primeiro?